# **ginger**

## **Setup**

In [ ]:
import sys
import os

def get_github_token():
    token = os.getenv("GITHUB_TOKEN")

    if token:
        return token

    import getpass
    return getpass.getpass("GitHub token: ")


try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
    print("Autoreload enabled.")
except Exception as e:
    print("Autoreload not available:", e)


if "google.colab" in sys.modules:
    print("Colab detected. Setting up repository...")

    %cd /content

    token = get_github_token()
    repo_url = f"https://{token}@github.com/Mango-Cats/ginger.git"

    if os.path.exists("ginger"):
        %cd ginger
        !git reset --hard HEAD
        !git pull origin main
    else:
        !git clone {repo_url}
        %cd ginger

    !pip install -e . --no-deps

else:
    print("Local environment detected.")

    if os.path.exists("ginger"):
        %cd ginger
    else:
        print("Repo not cloned locally. Set GITHUB_TOKEN or clone manually.")

import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(DEVICE)

Autoreload not available: No module named 'imp'
Google Colab detected. Setting up repository...
/content
/content/ginger
HEAD is now at ae826c2 Typo error fix
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 657 bytes | 328.00 KiB/s, done.
From https://github.com/Mango-Cats/ginger
 * branch            main       -> FETCH_HEAD
   ae826c2..229fc00  main       -> origin/main
Updating ae826c2..229fc00
Fast-forward
 ginger.ipynb   | 88 ++++++++++------------------------------------------------
 src/trainer.py | 11 ++++++--
 2 files changed, 22 insertions(+), 77 deletions(-)
Obtaining file:///content/ginger
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  

## **Data Loading & Preprocessing**

In [ ]:
from src.dataman import create_dataloaders
from src.dataman.constants import PAD_IDX
from pathlib import Path

In [4]:
DATA_PATH = Path("data/wikipron_tagalog.tsv")
BATCH_SIZE = 64

train_loader, val_loader, test_loader, grapheme_vocab, phoneme_vocab = create_dataloaders(
    DATA_PATH,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

print(f"Train samples: {len(train_loader.dataset)}")
print(f"Val samples: {len(val_loader.dataset)}")
print(f"Test samples: {len(test_loader.dataset)}")
print(f"Grapheme vocab size: {len(grapheme_vocab)}")
print(f"Phoneme vocab size: {len(phoneme_vocab)}")

Train samples: 14590
Val samples: 1824
Test samples: 1824
Grapheme vocab size: 32
Phoneme vocab size: 34


## **Model Initialization**

In [5]:
from src.architecture.transformer import Transformer
from torch.optim.lr_scheduler import OneCycleLR
from torch.optim import AdamW
from torch.amp import GradScaler
import torch.nn as nn

In [6]:
EMBEDDING_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ENCODER_LAYERS = 4
NUM_DECODER_LAYERS = 4
FEEDFORWARD_DIM = 512
DROPOUT_RATE = 0.1
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100 

model = Transformer(
    num_graphemes=len(grapheme_vocab),
    num_phonemes=len(phoneme_vocab),
    embedding_dim=EMBEDDING_DIM,
    num_attention_heads=NUM_ATTENTION_HEADS,
    num_encoder_layers=NUM_ENCODER_LAYERS,
    num_decoder_layers=NUM_DECODER_LAYERS,
    feedforward_dim=FEEDFORWARD_DIM,
    dropout_rate=DROPOUT_RATE,
).to(DEVICE)

model = torch.compile(model)

CRITERION = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
OPTIMIZER = AdamW(params=model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
SCHEDULER = OneCycleLR(
    optimizer=OPTIMIZER,
    max_lr=LEARNING_RATE,
    steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS,
    pct_start=0.1 
)
SCALER = torch.amp.GradScaler(DEVICE)
PATIENCE = 7

## **Training**

In [7]:
from src.trainer import train_model

In [8]:
results = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=CRITERION,
    optimizer=OPTIMIZER,
    scheduler=SCHEDULER,
    scaler=SCALER,
    device=DEVICE,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
)


Starting training for 100 epochs...



W0501 13:54:25.580000 29697 torch/_inductor/utils.py:1679] [1/0] Not enough SMs to use max_autotune_gemm mode


Epoch   5/100 | Train Loss: 0.9978 | Val Loss: 1.6957 (No improvement for 1 epochs) | Best (last 5): 1.5245 @ epoch 4
Epoch  10/100 | Train Loss: 0.7633 | Val Loss: 1.4567 (No improvement for 2 epochs) | Best (last 5): 1.4181 @ epoch 8
Epoch  15/100 | Train Loss: 0.5937 | Val Loss: 1.3894 (No improvement for 1 epochs) | Best (last 5): 1.2951 @ epoch 14
Epoch  20/100 | Train Loss: 0.4297 | Val Loss: 1.1824 (No improvement for 3 epochs) | Best (last 5): 1.0648 @ epoch 17
Epoch  25/100 | Train Loss: 0.3483 | Val Loss: 0.8997 (Best) | Best (last 5): 0.8997 @ epoch 25
Epoch  30/100 | Train Loss: 0.2894 | Val Loss: 1.1110 (No improvement for 5 epochs) | Best (last 5): 0.9326 @ epoch 28

Stopping early (patience=7)
Best validation loss: 0.8997
Restoring best model...


## **Inference**

In [11]:
from src.evaluation.evaluator import evaluate_model

In [12]:
eval_results = evaluate_model(
    model=model,
    test_loader=test_loader,
    grapheme_vocab=grapheme_vocab,
    phoneme_vocab=phoneme_vocab,
    device=DEVICE,
)


Metrics:
Total Words Tested: 1824
Phoneme Error Rate (PER): 18.20%
Word Error Rate (WER):    53.02%

Sample Predictions:

Word:     CARVAJAL
Expected: k a ɾ v a h a l
Output:   k a ɾ a v a h a l

Word:     ARAKIBOT
Expected: ʔ a ɾ a k i b o t
Output:   ʔ a b a ɾ k i t o

Word:     ORARYO
Expected: ʔ o ɾ a ɾ j o
Output:   ʔ o ɾ a j o ɾ

Word:     INANG
Expected: ʔ i n a ŋ
Output:   ʔ i n a ŋ

Word:     TIO
Expected: t j o
Output:   ʔ i t o
